# Part 8 — Full Model Comparison
**Appliance Energy Use Forecasting — 7PAM2033**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DolapoMichael/Time-Series-coding-Case-study-and-Report/blob/main/notebooks/08_model_comparison.ipynb)

Combines the saved outputs of Part 3 (benchmarks), Part 4 (SARIMAX), Part 6 (XGBoost), and Part 7 (Chronos-Bolt) into one comparison table and a shared set of plots.

**This notebook does not regenerate those parts** — it reads their saved `outputs/metrics/*.csv` and `outputs/forecasts/*.csv` files. Part 4's grid search alone takes 15-25 minutes and Part 7 needs a model download, so re-running them here on every open would be impractical. Run Parts 3, 4, 6, and 7 first in this same Colab session (or upload their `data/` and `outputs/` folders into a fresh session) before running this one. Any model whose files aren't found is skipped with a warning rather than failing the whole notebook, so partial comparisons still work.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.graphics.tsaplots import plot_acf

DATA_DIR = Path('data')
OUTPUT_DIR = Path('outputs')
for d in [OUTPUT_DIR / 'forecasts', OUTPUT_DIR / 'metrics', OUTPUT_DIR / 'figures']:
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})

## Load each part's saved outputs (skips gracefully if a part hasn't been run yet)

In [ ]:
sources = {
    'benchmarks': {
        'metrics': OUTPUT_DIR / 'metrics' / 'benchmark_metrics.csv',
        'forecasts': OUTPUT_DIR / 'forecasts' / 'benchmark_forecasts.csv',
    },
    'sarimax': {
        'metrics': OUTPUT_DIR / 'metrics' / 'sarimax_metrics.csv',
        'forecasts': OUTPUT_DIR / 'forecasts' / 'sarimax_forecasts.csv',
    },
    'xgboost': {
        'metrics': OUTPUT_DIR / 'metrics' / 'xgboost_metrics.csv',
        'forecasts': OUTPUT_DIR / 'forecasts' / 'xgboost_forecasts.csv',
    },
    'chronos': {
        'metrics': OUTPUT_DIR / 'metrics' / 'chronos_metrics.csv',
        'forecasts': OUTPUT_DIR / 'forecasts' / 'chronos_forecasts.csv',
    },
}

metrics_frames = []
forecast_frames = []
available = []

for name, paths in sources.items():
    if paths['metrics'].exists() and paths['forecasts'].exists():
        metrics_frames.append(pd.read_csv(paths['metrics']))
        fc = pd.read_csv(paths['forecasts'], index_col=0, parse_dates=True)
        forecast_frames.append(fc)
        available.append(name)
        print(f'Loaded {name}')
    else:
        print(f'SKIPPED {name} — run that part first (missing {paths["metrics"].name} or {paths["forecasts"].name})')

if not metrics_frames:
    raise FileNotFoundError('No model outputs found — run at least one of Parts 3/4/6/7 first.')

## Combine into one comparison table

In [ ]:
all_metrics = pd.concat(metrics_frames, ignore_index=True).sort_values('MASE').reset_index(drop=True)
all_metrics.to_csv(OUTPUT_DIR / 'metrics' / 'model_comparison.csv', index=False)
all_metrics.round(3)

## Combine forecasts into one table (one `actual` column, one column per model)

In [ ]:
all_forecasts = None
for fc in forecast_frames:
    fc = fc.copy()
    if all_forecasts is None:
        all_forecasts = fc
    else:
        # every source has its own 'actual' column (identical values) — keep one
        fc = fc.drop(columns=['actual'], errors='ignore')
        all_forecasts = all_forecasts.join(fc, how='outer')

all_forecasts.to_csv(OUTPUT_DIR / 'forecasts' / 'all_forecasts.csv')
print(f'Combined forecast table: {all_forecasts.shape}')
all_forecasts.head()

## Evaluation setup per model
Factual summary of how each model was actually evaluated — these are genuinely different setups (see Parts 4/5/6/7), not a level playing field.

In [ ]:
setup = pd.DataFrame([
    {'model_group': 'benchmarks (Part 3)', 'fitted_to_data': 'no (rule-based)', 'covariates_used': 'none', 'forecast_type': 'blind'},
    {'model_group': 'sarimax (Part 4)', 'fitted_to_data': 'yes', 'covariates_used': 'none', 'forecast_type': 'blind'},
    {'model_group': 'xgboost (Part 6)', 'fitted_to_data': 'yes', 'covariates_used': 'real historical lags + real future weather', 'forecast_type': 'conditional'},
    {'model_group': 'chronos (Part 7)', 'fitted_to_data': 'no (zero-shot)', 'covariates_used': 'none', 'forecast_type': 'blind'},
])
setup[setup['model_group'].str.split(' ').str[0].isin(available)]

## MASE comparison, all models

In [ ]:
strongest_benchmark_mase = all_metrics[all_metrics['model'].isin(
    ['mean', 'naive', 'seasonal_naive_daily', 'seasonal_naive_weekly', 'drift']
)]['MASE'].min() if any(all_metrics['model'].isin(['mean', 'naive', 'seasonal_naive_daily', 'seasonal_naive_weekly', 'drift'])) else None

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#c0392b' if m in ['mean', 'naive', 'seasonal_naive_daily', 'seasonal_naive_weekly', 'drift']
          else '#1f5b8a' for m in all_metrics['model']]
ax.barh(all_metrics['model'][::-1], all_metrics['MASE'][::-1], color=colors[::-1])
if strongest_benchmark_mase is not None:
    ax.axvline(strongest_benchmark_mase, color='black', linestyle='--', linewidth=1,
               label=f'Strongest benchmark (MASE={strongest_benchmark_mase:.3f})')
    ax.legend()
ax.set_xlabel('MASE (lower is better)')
ax.set_title('All models, sorted by MASE — red = benchmark, blue = fitted/foundation model')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures' / '08a_all_models_mase.png')
plt.show()

## Forecast comparison plot

In [ ]:
model_cols = [c for c in all_forecasts.columns if c != 'actual']

fig, ax = plt.subplots(figsize=(14, 5))
all_forecasts['actual'].plot(ax=ax, label='Actual', color='black', linewidth=1.8, zorder=10)
colors = plt.cm.tab10(np.linspace(0, 1, len(model_cols)))
for col, color in zip(model_cols, colors):
    all_forecasts[col].plot(ax=ax, label=col, alpha=0.75, linewidth=1, color=color)
ax.set_title('All models vs. actual — full 336h test period')
ax.set_xlabel('Date'); ax.set_ylabel('Appliances (Wh / hour)')
ax.legend(loc='upper right', fontsize=8, ncol=2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures' / 'forecast_comparison.png')
plt.show()

## Error diagnostics — residual distributions

In [ ]:
residuals = pd.DataFrame({col: all_forecasts[col] - all_forecasts['actual'] for col in model_cols})

fig, ax = plt.subplots(figsize=(10, 5))
residuals.boxplot(ax=ax, rot=30)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Residual (forecast − actual), Wh')
ax.set_title('Error distribution by model — full 336h test period')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures' / 'error_diagnostics.png')
plt.show()

## Residual autocorrelation — fitted/foundation models only
(benchmarks are rule-based, so residual ACF is less informative for them)

In [ ]:
diag_models = [c for c in ['sarimax', 'feature_model', 'foundation_model'] if c in residuals.columns]

if diag_models:
    fig, axes = plt.subplots(len(diag_models), 1, figsize=(10, 3 * len(diag_models)))
    axes = np.atleast_1d(axes)
    for ax, col in zip(axes, diag_models):
        plot_acf(residuals[col].dropna(), lags=48, ax=ax)
        ax.set_title(f'Residual ACF — {col}')
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'figures' / 'residual_acf.png')
    plt.show()
else:
    print('No fitted/foundation model outputs available yet for residual ACF.')

## Results

In [ ]:
print('Full comparison, sorted by MASE:')
all_metrics[['model', 'MAE', 'RMSE', 'MASE', 'Bias', 'n_points']].round(3)

Saved: `outputs/metrics/model_comparison.csv`, `outputs/forecasts/all_forecasts.csv`, and the four figures above (`08a_all_models_mase.png`, `forecast_comparison.png`, `error_diagnostics.png`, `residual_acf.png`).